<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_2_denoising_conv.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_2_denoising_conv.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F04_representation%2Fnotebooks%2Fsesion_04_2_denoising_conv.ipynb)

# SI7011 · Sesión 04 · 2. Denoising y autoencoders convolucionales
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 2** de las diapositivas, desde *Convolutional autoencoder*. En el notebook 1 el cuello de botella impedía que el autoencoder copiara la entrada. Aquí probamos otra forma de impedirlo: **corromper la entrada** y pedir la imagen limpia.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | tres formas de corromper $\mathbf x$ | qué recibe el encoder |
| 2 | autoencoder → *denoising* autoencoder, $k = 32$ | peor reconstrucción, mejor *probe* |
| 3 | sin cuello de botella: $k = 1024$ | copiar frente a aprender |
| 4 | capas lineales → convoluciones | menos parámetros, mejor reconstrucción; ¿mejor $\mathbf h$? |

**Datos:** Fashion-MNIST, como en el notebook 1. Corre en CPU en unos 10 minutos; con GPU, en 3. **En Kaggle:** agrega el dataset [`zalando-research/fashionmnist`](https://www.kaggle.com/datasets/zalando-research/fashionmnist) con **Add Input**.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import datasets

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· device', device)

### Cargar Fashion-MNIST
La celda busca los datos en este orden y usa el primero que encuentre. Imprime la **fuente** usada: revísala.

| Plataforma | De dónde salen los datos |
|---|---|
| **Kaggle** | el dataset [`zalando-research/fashionmnist`](https://www.kaggle.com/datasets/zalando-research/fashionmnist) agregado con **Add Input** (`/kaggle/input/...`), sin descargar |
| **Lightning Studio** o **local** | la carpeta `data/` junto al notebook, si ya tiene los archivos de una ejecución anterior |
| **Colab** | `kagglehub` con el mismo dataset y tu secreto `KAGGLE_API_TOKEN` (el mismo de la sesión 01) |
| sin dataset ni token | descarga con `torchvision` (unos 30 MB) |

Se leen los archivos originales (formato *idx*: una cabecera y luego los píxeles como bytes, imagen tras imagen). Si la copia solo trae los CSV (una fila por imagen: etiqueta y 784 píxeles), se leen esos.

In [ ]:
import gzip, os
from pathlib import Path
import numpy as np

KAGGLE_HANDLE = 'zalando-research/fashionmnist'      # archivos idx originales y CSV
ROOTS = [Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]
CLASSES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def read_idx(path):
    with (gzip.open if path.suffix == '.gz' else open)(path, 'rb') as f:
        raw = f.read()
    ndim = raw[3]                                        # 3: imágenes N × 28 × 28 · 1: etiquetas N
    dims = [int.from_bytes(raw[4 + 4 * i:8 + 4 * i], 'big') for i in range(ndim)]
    return np.frombuffer(raw, np.uint8, offset=4 + 4 * ndim).reshape(dims).copy()

def find_fmnist(roots):
    for root in [r for r in roots if r.exists()]:
        hit = next((p for p in sorted(root.rglob('train-images-idx3-ubyte*')) if p.is_file()), None)
        if hit is not None:
            return 'idx', hit.parent
        hit = next((p for p in root.rglob('fashion-mnist_train.csv') if p.is_file()), None)
        if hit is not None:
            return 'csv', hit.parent
    return None, None

def locate_fashion_mnist():
    kind, folder = find_fmnist(ROOTS)
    if folder is not None:
        return kind, folder, 'input de Kaggle' if str(folder).startswith('/kaggle/input') else 'archivos locales'
    if kaggle_token() or Path('/kaggle/input').exists():
        try:
            import kagglehub                               # preinstalado en Colab y Kaggle
            kind, folder = find_fmnist([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
            if folder is not None:
                return kind, folder, 'kagglehub'
        except Exception as e:
            print('kagglehub no disponible:', type(e).__name__, '→ se descarga con torchvision')
    from torchvision import datasets
    datasets.FashionMNIST('data', download=True)          # deja los idx en data/FashionMNIST/raw
    kind, folder = find_fmnist([Path('data')])
    return kind, folder, 'torchvision'

def load_fashion_mnist():
    kind, folder, source = locate_fashion_mnist()
    if kind == 'idx':
        f = lambda name: next(p for p in [folder / name, folder / (name + '.gz')] if p.exists())
        x_tr, y_tr = read_idx(f('train-images-idx3-ubyte')), read_idx(f('train-labels-idx1-ubyte'))
        x_te, y_te = read_idx(f('t10k-images-idx3-ubyte')), read_idx(f('t10k-labels-idx1-ubyte'))
    else:
        import pandas as pd
        tr, te = (pd.read_csv(folder / n).to_numpy() for n in ['fashion-mnist_train.csv', 'fashion-mnist_test.csv'])
        # cada fila: etiqueta, luego 784 píxeles → 28 × 28
        x_tr, y_tr = tr[:, 1:].reshape(-1, 28, 28).astype(np.uint8), tr[:, 0]
        x_te, y_te = te[:, 1:].reshape(-1, 28, 28).astype(np.uint8), te[:, 0]
    print('Fuente:', source, '·', folder)
    return x_tr, y_tr.astype(np.int64), x_te, y_te.astype(np.int64)

x_train_raw, y_train_raw, x_test_raw, y_test_raw = load_fashion_mnist()
x_train_raw.shape, x_test_raw.shape

In [ ]:
X_train = (torch.from_numpy(x_train_raw).float() / 255).unsqueeze(1).to(device)   # N × 1 × 28 × 28, valores en [0, 1]
X_test = (torch.from_numpy(x_test_raw).float() / 255).unsqueeze(1).to(device)
y_train, y_test = torch.from_numpy(y_train_raw).to(device), torch.from_numpy(y_test_raw).to(device)
X_train.shape

## Herramientas
Las del notebook 1, con un cambio: `train_ae` recibe una función `corrupt` y minimiza
$$L_i = \big\lVert \mathbf x_i - d\big(g(\tilde{\mathbf x}_i)\big)\big\rVert^2, \qquad \tilde{\mathbf x}_i = \text{corrupt}(\mathbf x_i).$$
El encoder ve la versión corrompida; el objetivo es la limpia. Con `corrupt = identidad` es el autoencoder del notebook 1. La corrupción es nueva en cada paso, como Dropout en la sesión 02.

In [ ]:
def train_ae(g, d, corrupt=lambda x: x, steps=3000, batch=128, lr=1e-3):
    g.to(device); d.to(device)
    opt = torch.optim.Adam([*g.parameters(), *d.parameters()], lr=lr)
    step, t0 = 0, time.time()
    while step < steps:
        for b in torch.randperm(len(X_train), device=device).split(batch):
            xb = X_train[b]
            loss = ((d(g(corrupt(xb))) - xb) ** 2).mean()      # x̃ entra al encoder; el objetivo es x limpio
            opt.zero_grad(); loss.backward(); opt.step()
            step += 1
            if step == steps: break
    print(f'{steps} pasos en {time.time() - t0:.0f} s')

def medir(g, d, corrupt, chunk=2000):
    torch.manual_seed(1)
    with torch.no_grad():
        rec = lambda f: torch.cat([((d(g(f(X_test[i:i + chunk]))) - X_test[i:i + chunk]) ** 2).mean((1, 2, 3))
                                   for i in range(0, len(X_test), chunk)]).mean().item()
        m_limpio, m_corrupto = rec(lambda x: x), rec(corrupt)
        H_tr = torch.cat([g(X_train[i:i + chunk]).flatten(1) for i in range(0, len(X_train), chunk)])
        H_te = torch.cat([g(X_test[i:i + chunk]).flatten(1) for i in range(0, len(X_test), chunk)])
    # MSE con entrada limpia, MSE con entrada corrompida, y probe sobre h = g(x) de imágenes limpias
    return {'mse_limpio': m_limpio, 'mse_corrupto': m_corrupto, 'probe': probe(H_tr, H_te)}

def probe(H_train, H_test, steps=500):
    mu, sd = H_train.mean(0), H_train.std(0) + 1e-6
    Htr, Hte = (H_train - mu) / sd, (H_test - mu) / sd
    torch.manual_seed(0)
    head = nn.Linear(Htr.shape[1], 10).to(device)        # la única pieza que ve etiquetas
    opt = torch.optim.Adam(head.parameters(), lr=1e-2)
    for _ in range(steps):
        loss = nn.functional.cross_entropy(head(Htr), y_train)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head(Hte).argmax(1) == y_test).float().mean().item()

def mlp_ae(k, hidden=256):
    g = nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU(), nn.Linear(hidden, k))
    d = nn.Sequential(nn.Linear(k, hidden), nn.ReLU(), nn.Linear(hidden, 784), nn.Unflatten(1, (1, 28, 28)))
    return g, d

print(f"probe sobre los píxeles: {probe(X_train.flatten(1), X_test.flatten(1)):.3f}")

## 1. Tres formas de corromper $\mathbf x$
- **Ruido gaussiano:** $\tilde{\mathbf x} = \mathbf x + \boldsymbol\varepsilon$, con $\boldsymbol\varepsilon \sim \mathcal N(0, \sigma^2 I)$.
- **Máscara:** cada píxel se apaga con probabilidad $p$, y los que quedan se multiplican por $1/(1-p)$, igual que en Dropout. Así la imagen limpia y la enmascarada tienen el mismo brillo promedio.

In [ ]:
gauss = lambda s: (lambda x: x + s * torch.randn_like(x))                                # x + ε
mascara = lambda p: (lambda x: x * (torch.rand_like(x) > p) / (1 - p))                   # Dropout sobre píxeles

torch.manual_seed(0)
ejemplos = X_test[:6]
filas = {'limpia': ejemplos, 'gauss σ = 0.3': gauss(0.3)(ejemplos), 'gauss σ = 0.6': gauss(0.6)(ejemplos),
         'máscara p = 0.5': mascara(0.5)(ejemplos)}
fig, axes = plt.subplots(len(filas), 6, figsize=(7, 4.8))
for r, (n, im) in enumerate(filas.items()):
    for c in range(6):
        axes[r, c].imshow(im[c, 0].cpu(), cmap='gray_r', vmin=0, vmax=1); axes[r, c].axis('off')
    axes[r, 0].set_title(n, fontsize=8, loc='left')
plt.tight_layout(); plt.show()

## 2. Autoencoder frente a *denoising* autoencoder, $k = 32$
**Cambia solo la entrada del encoder:** la misma red del notebook 1, una vez con $\mathbf x$ limpio y otra con $\tilde{\mathbf x} = \mathbf x + \boldsymbol\varepsilon$, $\sigma = 0.3$.

**Predice:** ¿cuál reconstruye mejor una imagen **limpia**? ¿Cuál tiene mejor *probe*?

In [ ]:
res = {}
torch.manual_seed(0); g_ae, d_ae = mlp_ae(32)
train_ae(g_ae, d_ae)
res['AE'] = medir(g_ae, d_ae, gauss(0.3))

torch.manual_seed(0); g_dae, d_dae = mlp_ae(32)
train_ae(g_dae, d_dae, corrupt=gauss(0.3))                 # NUEVO: el encoder ve x + ε
res['DAE σ=0.3'] = medir(g_dae, d_dae, gauss(0.3))
for n, r in res.items():
    print(f"{n:10s} · MSE limpio {r['mse_limpio']:.4f} · MSE desde x̃ {r['mse_corrupto']:.4f} · probe {r['probe']:.3f}")

In [ ]:
torch.manual_seed(2)
x = X_test[10:16]; xt = gauss(0.3)(x)
with torch.no_grad():
    filas = {'x limpia': x, 'x̃ = x + ε': xt, 'AE(x̃)': d_ae(g_ae(xt)), 'DAE(x̃)': d_dae(g_dae(xt))}
fig, axes = plt.subplots(4, 6, figsize=(7, 4.8))
for r, (n, im) in enumerate(filas.items()):
    for c in range(6):
        axes[r, c].imshow(im[c, 0].cpu().clamp(0, 1), cmap='gray_r', vmin=0, vmax=1); axes[r, c].axis('off')
    axes[r, 0].set_title(n, fontsize=8, loc='left')
plt.tight_layout(); plt.show()

El *denoising* autoencoder reconstruye **peor** una imagen limpia: nunca la vio durante el entrenamiento. Pero limpia mejor el ruido, y su *probe* sube. Para quitar el ruido tuvo que aprender cómo son las prendas, no solo cómo copiar píxeles. Es otra vez la diapositiva *Reconstruction is not the goal*.

## 3. Sin cuello de botella: $k = 1024$
En el notebook 1, con $k = 784$ el autoencoder copiaba la entrada y $\mathbf h$ no aportaba nada. Ahora el código es **más ancho** que la imagen: $g(\mathbf x) = \text{ReLU}(W_g\mathbf x + \mathbf b)$ con $W_g \in \mathbb R^{1024 \times 784}$, y $d$ lineal.

**Predice:** sin corrupción, ¿la *probe* sobre $\mathbf h$ le gana a la *probe* sobre los píxeles? ¿Y con corrupción?

In [ ]:
def ancho(k=1024):
    g = nn.Sequential(nn.Flatten(), nn.Linear(784, k), nn.ReLU())                  # h ∈ ℝ^1024: más grande que x
    d = nn.Sequential(nn.Linear(k, 784), nn.Unflatten(1, (1, 28, 28)))
    return g, d

anchos = {}
for nombre, corr in [('sin corrupción', lambda x: x), ('gauss σ = 0.5', gauss(0.5)), ('máscara p = 0.5', mascara(0.5))]:
    torch.manual_seed(0); g, d = ancho()
    train_ae(g, d, corrupt=corr)
    anchos[nombre] = (g, d, medir(g, d, corr))
    r = anchos[nombre][2]
    print(f"k = 1024 · {nombre:16s} · MSE limpio {r['mse_limpio']:.4f} · probe {r['probe']:.3f}")

Sin corrupción, el error es el más bajo y la *probe* queda donde estaba la de los píxeles: con espacio de sobra, la red copia. Con ruido o con máscara, el mismo $\mathbf h$ de 1024 números le gana a los píxeles, porque copiar ya no sirve para bajar la pérdida. La diapositiva *Denoising autoencoder* lo resume: **cuando no se puede copiar, el cuello de botella puede ser ancho.**

Cada fila de $W_g$ es un detector de 784 pesos y se puede ver como una imagen de 28×28.

In [ ]:
fig, axes = plt.subplots(3, 12, figsize=(12, 3.3))
for r, (nombre, (g, d, _)) in enumerate(anchos.items()):
    W = g[1].weight.detach().cpu()
    top = W.norm(dim=1).argsort(descending=True)[:12]       # las 12 unidades con pesos más grandes
    for c, i in enumerate(top):
        axes[r, c].imshow(W[i].reshape(28, 28), cmap='RdBu_r'); axes[r, c].axis('off')
    axes[r, 0].set_title(nombre, fontsize=8, loc='left')
plt.tight_layout(); plt.show()

Compara las filas. Sin corrupción, las unidades parecen ruido: para copiar, cualquier mezcla de píxeles sirve. Con ruido gaussiano aparecen trazos y bordes (la suela de un zapato, la tira de una sandalia); con máscara, siluetas casi completas de camisas y pantalones. En los dos casos son detectores que **combinan** muchos píxeles, porque un píxel aislado ya no es confiable.

## 4. Un autoencoder convolucional
**Cambia las capas:** el encoder es una CNN pequeña de la sesión 03 (dos convoluciones con *stride* 2: 28 → 14 → 7) y el decoder hace el camino inverso con `ConvTranspose2d`. El código sigue siendo $\mathbf h \in \mathbb R^{32}$ y se entrena como *denoising* autoencoder, para compararlo con el de la sección 2.

In [ ]:
def conv_ae(k=32):
    g = nn.Sequential(
        nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),                 # 1×28×28 → 32×14×14
        nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),                # → 64×7×7
        nn.Flatten(), nn.Linear(64 * 7 * 7, k))                             # → h ∈ ℝ^k
    d = nn.Sequential(
        nn.Linear(k, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
        nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),   # → 32×14×14
        nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1))               # → 1×28×28
    return g, d

n_par = lambda *ms: sum(p.numel() for m in ms for p in m.parameters())
torch.manual_seed(0); g_c, d_c = conv_ae(32)
print(f'parámetros · MLP {n_par(g_dae, d_dae):,} · convolucional {n_par(g_c, d_c):,}')
train_ae(g_c, d_c, corrupt=gauss(0.3))
res['DAE conv σ=0.3'] = medir(g_c, d_c, gauss(0.3))
for n, r in res.items():
    print(f"{n:15s} · MSE limpio {r['mse_limpio']:.4f} · MSE desde x̃ {r['mse_corrupto']:.4f} · probe {r['probe']:.3f}")

Con cerca de un 40 % menos de parámetros, el autoencoder convolucional reconstruye mejor, desde la imagen limpia y desde la ruidosa: los supuestos de la sesión 03 (localidad y pesos compartidos) también sirven cuando no hay etiquetas. Pero su *probe* queda prácticamente igual a la del MLP. Con un $\mathbf h$ de 32 números, en imágenes tan simples, lo que limita la *probe* es el objetivo, no la arquitectura. Ese es el tema de la sesión: lo que cambia entre la sesión 03 y esta es el **objetivo**, y la Parte 4 cambia el objetivo por completo.

## Tu turno · 10 minutos
1. En la sección 2, sube el ruido a $\sigma = 1.0$. ¿Sigue mejorando la *probe*? ¿Hay un punto en que la tarea se vuelve imposible?
2. En la sección 3, prueba la máscara con $p = 0.25$ y $p = 0.75$. ¿Cuál deja la mejor *probe*? Guarda tu respuesta para el notebook 3, donde se enmascara el 75 % de la imagen.
3. Entrena el autoencoder convolucional **sin** corrupción. ¿Cuánto de su ventaja venía de las convoluciones y cuánto del ruido?

**Para pensar.** El *denoising* autoencoder aprende a quitar un ruido que nosotros elegimos. ¿Qué pasaría si en producción las imágenes llegaran con otro tipo de daño (borrosas, recortadas)? ¿Qué supuesto se esconde en la elección de la corrupción? (pista: diapositiva *What each objective assumes*).

In [ ]:
# Experimento 1: σ = 1.0 en la sección 2

### Referencias
- Vincent, Larochelle, Bengio y Manzagol (2008), *Extracting and Composing Robust Features with Denoising Autoencoders*, ICML.
- Vincent et al. (2010), *Stacked Denoising Autoencoders*, JMLR.
- Masci, Meier, Cireşan y Schmidhuber (2011), *Stacked Convolutional Auto-Encoders for Hierarchical Feature Extraction*, ICANN.